# CNN-EXP-047 — paired WIDTH512 risk-focused loss continuation

## Pre-registration

**Question:** can end-to-end emphasis on high-scoring train negatives produce a material improvement in natural genome-wide nucleotide AP over an otherwise identical continuation?

- Branch point: EXP045 WIDTH512 at step 50,000.
- A: matched continuation with the original class-balanced presence BCE.
- B: identical continuation; half of the negative objective is redistributed equally across four fixed EXP045 score strata: top 1%, 1–5%, 5–15%, bottom 85%.
- Architecture, sampler, batch order, Adam moments, RNG, intensity loss and LR are identical between A and B.
- Validation is not used to define strata or choose a checkpoint. Each final branch receives one full natural validation scan.
- `ADOPT` requires B−A AP ≥ 0.02 and positive B−A AP on every validation contig.

The notebook is restart-safe. Long DirectML stages run in isolated child processes. Re-running a cell either resumes from the atomic `latest.pt` or skips a completed stage.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd
from IPython.display import display, Image

EXPECTED_PYTHON = Path(r"C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe").resolve()
assert Path(sys.executable).resolve() == EXPECTED_PYTHON, (
    f"Wrong kernel: {sys.executable}. Select the epic-eda kernel: {EXPECTED_PYTHON}"
)
CURRENT = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (CURRENT, *CURRENT.parents)
    if (candidate / "README.md").is_file() and (candidate / "src/ml_project").is_dir()
)
SCRIPT = PROJECT_ROOT / "scripts/run_cnn_exp047_paired_risk_loss.py"
RUN_DIR = PROJECT_ROOT / "artifacts/experiments/CNN_EXP047_WIDTH512_PAIRED_RISK_LOSS_CONTINUATION/run_001"
assert SCRIPT.is_file(), SCRIPT

def run_stage(*arguments):
    command = [str(EXPECTED_PYTHON), "-u", str(SCRIPT), *map(str, arguments)]
    print("RUN:", subprocess.list2cmdline(command), flush=True)
    process = subprocess.Popen(
        command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1,
    )
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
    except KeyboardInterrupt:
        process.terminate()
        process.wait(timeout=30)
        raise
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Stage failed with exit code {return_code}: {command}")

print("Python:", sys.executable)
print("Project:", PROJECT_ROOT)
print("Artifacts:", RUN_DIR)

Python: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe
Project: D:\Projects\EPIC\EPIC
Artifacts: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP047_WIDTH512_PAIRED_RISK_LOSS_CONTINUATION\run_001


## 1. Seal the protocol and inspect all inputs

This must run before cache construction or training. An existing changed pre-registration is never overwritten.

In [2]:
run_stage("preregister")
run_stage("inspect")
display(pd.DataFrame([json.loads((RUN_DIR / "EXP047_preregistration.json").read_text(encoding="utf-8"))]))

RUN: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe -u D:\Projects\EPIC\EPIC\scripts\run_cnn_exp047_paired_risk_loss.py preregister
Pre-registration: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP047_WIDTH512_PAIRED_RISK_LOSS_CONTINUATION\run_001\EXP047_preregistration.json
Experiment card: D:\Projects\EPIC\EPIC\cnn_exp\CNN-EXP-047.md
RUN: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe -u D:\Projects\EPIC\EPIC\scripts\run_cnn_exp047_paired_risk_loss.py inspect
Python: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe
Base checkpoint: step=50000, sha256=9e6b5b9a4fa3b9969fc7e5441f857ff6faa831adf4e4e753172db18d26957982
Model parameters: 14,708,738
Corrected train shards: folds=9, files=8,161
LR probes: step1=3.014e-05, warmup=0.0001, final=1e-05


,experiment_id,created,status,primary_question,branch_point,arms,negative_strata,stratum_source,candidate_negative_mixture,frozen_between_arms,continuation,primary_metric,selection_rule,validation_policy,checkpoint_policy,scientific_caveat
0,CNN-EXP-047,2026-09-27,pre_registered,Does end-to-end risk-focused presence training...,D:\Projects\EPIC\EPIC\artifacts\experiments\CN...,{'A': 'matched continuation with the original ...,"[top_0_1pct, top_1_5pct, top_5_15pct, bottom_8...",EXP045 train-negative logits already saved in ...,"[0.13, 0.145, 0.175, 0.55]","[initial model, Adam moments, generator RNG, t...","{'steps': 20000, 'batch_size': 4, 'warmup_step...",natural full-validation nucleotide AP after 5-...,ADOPT only if B-A overall AP >= 0.02 and B imp...,one full final scan per arm; no checkpoint sel...,atomic latest.pt every 500 steps; immutable sn...,The validation split has been inspected in pri...


## 2. Build fixed train-negative danger strata

This is CPU-only and reads the corrected EXP046B shards generated from EXP045 on the **nine train contigs**. It creates a compact `uint8` lookup by `(tile_id, strand, position)`. Positive and invalid positions receive code 255. Ties at a 5-decimal score boundary are resolved deterministically. Validation is not read.

In [3]:
run_stage("prepare-risk-cache")
risk_manifest = json.loads((RUN_DIR / "risk_strata_cache/risk_strata_manifest.json").read_text(encoding="utf-8"))
display(pd.DataFrame({
    "stratum": ["top_0_1pct", "top_1_5pct", "top_5_15pct", "bottom_85pct"],
    "count": risk_manifest["stratum_counts"],
    "natural_fraction": risk_manifest["natural_fractions"],
    "B_weight_multiplier": risk_manifest["candidate_negative_weight_multipliers"],
}))
run_stage("smoke", "--branch", "A")
run_stage("smoke", "--branch", "B")

RUN: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe -u D:\Projects\EPIC\EPIC\scripts\run_cnn_exp047_paired_risk_loss.py prepare-risk-cache
Complete risk cache already exists: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP047_WIDTH512_PAIRED_RISK_LOSS_CONTINUATION\run_001\risk_strata_cache\train_negative_stratum_u8.dat


,stratum,count,natural_fraction,B_weight_multiplier
0,top_0_1pct,2278937,0.01,13.000001
1,top_1_5pct,9115749,0.04,3.625000
2,top_5_15pct,22789372,0.10,1.750000
3,bottom_85pct,193709659,0.85,0.647059


RUN: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe -u D:\Projects\EPIC\EPIC\scripts\run_cnn_exp047_paired_risk_loss.py smoke --branch A
SMOKE OK branch=A device=AMD Radeon RX 6750 GRE 12GB  presence=0.106830 intensity=0.178568 total=0.124687 atomic_checkpoint=OK
RUN: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe -u D:\Projects\EPIC\EPIC\scripts\run_cnn_exp047_paired_risk_loss.py smoke --branch B
SMOKE OK branch=B device=AMD Radeon RX 6750 GRE 12GB  presence=0.379205 intensity=0.178568 total=0.397061 atomic_checkpoint=OK


## 3. Branch A — matched control continuation

The original presence/intensity objective continues for 20,000 steps. `latest.pt` is committed atomically every 500 steps; immutable snapshots are retained at 2k, 5k, 10k, 15k and 20k. Re-run this cell after a crash to resume.

In [4]:
run_stage("train", "--branch", "A")

RUN: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe -u D:\Projects\EPIC\EPIC\scripts\run_cnn_exp047_paired_risk_loss.py train --branch A
[A] local=00250/20000 global=50250 lr=6.5e-05 loss=(0.09868,0.14801,0.11348) sec/step=0.237 ETA=1.30h
[A] local=00500/20000 global=50500 lr=0.0001 loss=(0.36769,0.44312,0.41201) sec/step=0.232 ETA=1.26h
[A] local=00750/20000 global=50750 lr=9.99635e-05 loss=(0.33113,0.16554,0.34768) sec/step=0.231 ETA=1.24h
[A] local=01000/20000 global=51000 lr=9.985408e-05 loss=(0.12066,0.39439,0.16010) sec/step=0.230 ETA=1.21h
[A] local=01250/20000 global=51250 lr=9.96719e-05 loss=(0.18814,0.10376,0.19852) sec/step=0.230 ETA=1.20h
[A] local=01500/20000 global=51500 lr=9.941726e-05 loss=(0.25393,0.04857,0.25879) sec/step=0.230 ETA=1.18h
[A] local=01750/20000 global=51750 lr=9.909058e-05 loss=(0.34871,0.11983,0.36069) sec/step=0.230 ETA=1.16h
[A] local=02000/20000 global=52000 lr=9.869238e-05 loss=(0.25586,0.05332,0.26119) sec/step=0.229 ETA=1.15h
[A] local=

## 4. Branch B — risk-focused continuation

Starts independently from the same EXP045 checkpoint, Adam state and RNG. The only A/B change is the negative presence-loss weighting.

In [5]:
run_stage("train", "--branch", "B")

RUN: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe -u D:\Projects\EPIC\EPIC\scripts\run_cnn_exp047_paired_risk_loss.py train --branch B
[B] local=00250/20000 global=50250 lr=6.5e-05 loss=(0.21397,0.14566,0.22854) sec/step=0.238 ETA=1.30h
[B] local=00500/20000 global=50500 lr=0.0001 loss=(0.52367,0.44473,0.56815) sec/step=0.232 ETA=1.26h
[B] local=00750/20000 global=50750 lr=9.99635e-05 loss=(0.56336,0.17266,0.58063) sec/step=0.232 ETA=1.24h
[B] local=01000/20000 global=51000 lr=9.985408e-05 loss=(0.28751,0.39146,0.32666) sec/step=0.231 ETA=1.22h
[B] local=01250/20000 global=51250 lr=9.96719e-05 loss=(0.48973,0.11309,0.50104) sec/step=0.230 ETA=1.20h
[B] local=01500/20000 global=51500 lr=9.941726e-05 loss=(0.43517,0.03918,0.43908) sec/step=0.230 ETA=1.18h
[B] local=01750/20000 global=51750 lr=9.909058e-05 loss=(0.51658,0.12402,0.52899) sec/step=0.230 ETA=1.17h
[B] local=02000/20000 global=52000 lr=9.869238e-05 loss=(0.44989,0.05788,0.45568) sec/step=0.230 ETA=1.15h
[B] local=

## 5. One natural full-validation scan per final branch

No intermediate checkpoint is selected on validation. Each evaluation also rescans the fixed EXP046 HARD_FP and LOW_TP coordinates to reveal whether score changes occurred in the intended populations.

In [6]:
run_stage("evaluate", "--branch", "A")
run_stage("evaluate", "--branch", "B")

RUN: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe -u D:\Projects\EPIC\EPIC\scripts\run_cnn_exp047_paired_risk_loss.py evaluate --branch A
tiles=     1/47743 positions=      210 positions/s=1,510
tiles=   100/47743 positions=   125828 positions/s=52,971
tiles=   200/47743 positions=   269398 positions/s=58,197
tiles=   300/47743 positions=   403094 positions/s=58,449
tiles=   400/47743 positions=   552594 positions/s=60,338
tiles=   500/47743 positions=   708204 positions/s=62,043
tiles=   600/47743 positions=   871458 positions/s=63,759
tiles=   700/47743 positions=  1020090 positions/s=64,080
tiles=   800/47743 positions=  1185336 positions/s=65,185
tiles=   900/47743 positions=  1346300 positions/s=65,861
tiles=  1000/47743 positions=  1519462 positions/s=66,928
tiles=  1100/47743 positions=  1700978 positions/s=68,135
tiles=  1200/47743 positions=  1853300 positions/s=68,076
tiles=  1300/47743 positions=  2017896 positions/s=68,435
tiles=  1400/47743 positions=  2193246 

## 6. Finalize artifacts and apply the pre-registered decision rule

This writes the compact result JSON, paired tables, plots, experiment card and CNN registry entry.

In [7]:
run_stage("finalize")
result = json.loads((RUN_DIR / "EXP047_RESULT.json").read_text(encoding="utf-8"))
comparison = pd.read_csv(RUN_DIR / "paired_validation_comparison.csv")
display(comparison)
cohort_summary = RUN_DIR / "fixed_cohort_A_vs_B_summary.csv"
if cohort_summary.is_file():
    display(pd.read_csv(cohort_summary))
display(result)
for image_name in ("04_paired_training_losses.png", "05_exp045_vs_A_vs_B_ap.png"):
    image_path = RUN_DIR / "plots" / image_name
    if image_path.is_file():
        display(Image(filename=str(image_path)))

RUN: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe -u D:\Projects\EPIC\EPIC\scripts\run_cnn_exp047_paired_risk_loss.py finalize
Traceback (most recent call last):
  File "D:\Projects\EPIC\EPIC\scripts\run_cnn_exp047_paired_risk_loss.py", line 1269, in <module>
    main()
  File "D:\Projects\EPIC\EPIC\scripts\run_cnn_exp047_paired_risk_loss.py", line 1263, in main
    finalize()
  File "D:\Projects\EPIC\EPIC\scripts\run_cnn_exp047_paired_risk_loss.py", line 1090, in finalize
    preregister()
  File "D:\Projects\EPIC\EPIC\scripts\run_cnn_exp047_paired_risk_loss.py", line 327, in preregister
    raise RuntimeError(f"Refusing to overwrite changed pre-registration: {PREREG_PATH}")
RuntimeError: Refusing to overwrite changed pre-registration: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP047_WIDTH512_PAIRED_RISK_LOSS_CONTINUATION\run_001\EXP047_preregistration.json


RuntimeError: Stage failed with exit code 1: ['C:\\ML_Progs\\Runtimes\\Miniforge3\\envs\\epic-eda\\python.exe', '-u', 'D:\\Projects\\EPIC\\EPIC\\scripts\\run_cnn_exp047_paired_risk_loss.py', 'finalize']

## Expected durable outputs

- `EXP047_preregistration.json`
- `risk_strata_cache/risk_strata_manifest.json` and the `uint8` lookup
- `branch_A/` and `branch_B/`: atomic resume checkpoint, immutable checkpoints, history, final model and validation
- `paired_training_history.csv` and `paired_validation_comparison.csv`
- `fixed_cohort_A_vs_B.csv` and its summary
- `validation_metrics_report.csv`, `strong_validation_metrics.csv`
- `EXP047_RESULT.json`, plots, Obsidian card and registry update